#  LEO Constellation Generation

This section generates a dynamic snapshot of a LEO satellite constellation
based on orbital parameters and simulation time.

In [ ]:
import heapq
import numpy as np
from collections import defaultdict
from scipy.spatial import distance, KDTree
import pandas as pd
from scipy.special import erfc
from scipy.integrate import quad
from sklearn.decomposition import PCA

EARTH_RADIUS = 6371  # km
def rotate_all_orbits_by_plane(time_minutes=0, num_orbits=24, sats_per_orbit=66, orbital_period=5400):

    import numpy as np
    import pandas as pd

    EARTH_RADIUS = 6371  # km
    altitude = 550  # km
    semi_major_axis = EARTH_RADIUS + altitude  # km
    mean_motion = 2 * np.pi / orbital_period  # rad/s
    delta_t = time_minutes * 60
    inclination = np.radians(53.0)
    eccentricity = 0.001
    rows = []
    counter = 0
    for orbit_id in range(num_orbits):
        raan = np.radians(orbit_id * (360.0 / num_orbits))
        for rel_id in range(sats_per_orbit):

            mean_anomaly_initial = rel_id * (2 * np.pi / sats_per_orbit)

            mean_anomaly = mean_anomaly_initial + mean_motion * delta_t
            mean_anomaly = mean_anomaly % (2 * np.pi)  # نرمال‌سازی

            eccentric_anomaly = mean_anomaly + eccentricity * np.sin(mean_anomaly)

            true_anomaly = 2 * np.arctan2(
                np.sqrt(1 + eccentricity) * np.sin(eccentric_anomaly / 2),
                np.sqrt(1 - eccentricity) * np.cos(eccentric_anomaly / 2)
            )

            r = semi_major_axis * (1 - eccentricity * np.cos(eccentric_anomaly))


            x_orbital = r * np.cos(true_anomaly)
            y_orbital = r * np.sin(true_anomaly)

            x_earth = (x_orbital * np.cos(raan) -
                      y_orbital * np.cos(inclination) * np.sin(raan))
            y_earth = (x_orbital * np.sin(raan) +
                      y_orbital * np.cos(inclination) * np.cos(raan))
            z_earth = y_orbital * np.sin(inclination)

            lat_rad = np.arcsin(z_earth / r)
            lon_rad = np.arctan2(y_earth, x_earth)

            lat_deg = np.degrees(lat_rad)
            lon_deg = np.degrees(lon_rad)

            lon_deg = (lon_deg + 180) % 360 - 180

            alt_km = r - EARTH_RADIUS

            rows.append([counter, orbit_id, rel_id, lat_deg, lon_deg, alt_km])
            counter += 1

    df = pd.DataFrame(rows, columns=["sat_id", "orbit_id", "rel_id_in_orbit",
                                     "lat_deg", "lon_deg", "alt_km"])

    return df
def spherical_to_cartesian(lat_deg, lon_deg, alt_km):
    lat_rad = np.radians(lat_deg)
    lon_rad = np.radians(lon_deg)
    r = EARTH_RADIUS + alt_km
    x = r * np.cos(lat_rad) * np.cos(lon_rad)
    y = r * np.cos(lat_rad) * np.sin(lon_rad)
    z = r * np.sin(lat_rad)
    return np.column_stack((x, y, z))



#  Inter-Satellite Link Construction

This section builds intra-orbit, inter-orbit, and temporary inter-satellite links (TLs).

In [ ]:
def find_intra_orbit_links(df, positions, sats_per_orbit=66):
    intra_links = {}
    total_sats = len(df)
    for i in range(total_sats):
        sat_id = int(df.iloc[i]['sat_id'])
        orbit_start = (i // sats_per_orbit) * sats_per_orbit
        local_idx = i % sats_per_orbit
        neighbors = [
            orbit_start + (local_idx - 2) % sats_per_orbit,
            orbit_start + (local_idx - 1) % sats_per_orbit,
            orbit_start + (local_idx + 1) % sats_per_orbit,
            orbit_start + (local_idx + 2) % sats_per_orbit
        ]
        intra_links[sat_id] = []
        for j in neighbors:
            if j < total_sats:
                neighbor_id = int(df.iloc[j]['sat_id'])
                dist = np.linalg.norm(positions[i] - positions[j])
                intra_links[sat_id].append((neighbor_id, dist))
    return intra_links

def find_inter_orbit_links_optimized(df, positions, sats_per_orbit=66, max_tl_distance=1500):
    inter_links = {}
    total_sats = len(df)
    num_orbits = total_sats // sats_per_orbit

    orbit_positions = []
    for i in range(num_orbits):
        start = i * sats_per_orbit
        end = start + sats_per_orbit
        orbit_positions.append(positions[start:end])

    for i in range(total_sats):
        sat_id = int(df.iloc[i]['sat_id'])
        orbit_idx = i // sats_per_orbit
        target_pos = positions[i]
        inter_links[sat_id] = []

        for orb_offset in [-1, 1]:
            neighbor_orbit = (orbit_idx + orb_offset) % num_orbits
            neighbor_positions = orbit_positions[neighbor_orbit]

            distances = np.linalg.norm(neighbor_positions - target_pos, axis=1)

            if max_tl_distance == 1500:
                min_idx = np.argmin(distances)
                neighbor_global_idx = neighbor_orbit * sats_per_orbit + min_idx
                neighbor_id = int(df.iloc[neighbor_global_idx]['sat_id'])
                dist = distances[min_idx]
                inter_links[sat_id].append((neighbor_id, dist))

            elif max_tl_distance == 1700:
                nearest_indices = np.argsort(distances)[:3]
                for idx in nearest_indices:
                    neighbor_global_idx = neighbor_orbit * sats_per_orbit + idx
                    neighbor_id = int(df.iloc[neighbor_global_idx]['sat_id'])
                    dist = distances[idx]
                    inter_links[sat_id].append((neighbor_id, dist))

            else:
                raise ValueError("Only LISL ranges 1500 or 1700 km are supported for inter-links")

    return inter_links


def find_temporary_links_optimized(df, positions, intra_links, inter_links, max_distance=1500,num=5):
    tl_links = {}
    kd_tree = KDTree(positions)
    sat_ids = df['sat_id'].values
    permanent_links = {}
    for sid in sat_ids:
        intra_neighbors = [n[0] for n in intra_links.get(int(sid), [])]
        inter_neighbors = [n[0] for n in inter_links.get(int(sid), [])]
        permanent_links[int(sid)] = set(intra_neighbors + inter_neighbors)
    for i, sid in enumerate(sat_ids):
        sid_int = int(sid)
        neighbors = kd_tree.query_ball_point(positions[i], max_distance)
        candidates = []
        for j in neighbors:
            if i == j:
                continue
            nid = int(sat_ids[j])
            if nid not in permanent_links[sid_int]:
                dist = np.linalg.norm(positions[i] - positions[j])
                candidates.append((nid, dist))

        candidates.sort(key=lambda x: x[1])
        tl_links[sid_int] = candidates[:num]

    return tl_links

#  Graph Construction & Propagation Delay Modeling

This section converts the satellite network into a weighted graph
based on propagation delay.

In [ ]:
def build_satellite_network_dynamic(df, positions, max_tl_distance=1500, sats_per_orbit=66, num=5):
    intra_links = find_intra_orbit_links(df, positions, sats_per_orbit)
    inter_links = find_inter_orbit_links_optimized(df, positions, sats_per_orbit, max_tl_distance=max_tl_distance)
    tl_links = find_temporary_links_optimized(df, positions, intra_links, inter_links, max_tl_distance, num)
    return {
        'nodes': [{
            'id': int(row['sat_id']),
            'position': positions[i],
            'orbit_id': int(row['orbit_id']),
            'rel_id_in_orbit': int(row['rel_id_in_orbit'])
        } for i, row in df.iterrows()],
        'links': {
            'intra_orbit': intra_links,
            'inter_orbit': inter_links,
            'temporary': tl_links
        },
        'statistics': {
            'total_nodes': len(df),
            'total_intra_links': sum(len(v) for v in intra_links.values()),
            'total_inter_links': sum(len(v) for v in inter_links.values()),
            'total_tl_links': sum(len(v) for v in tl_links.values())
        }
    }
def build_weighted_graph(
    network,
    c_km_s=299792.458,
    node_delay_const=0.004,
    setup_delay_tl=0.005,
    active_tl_links=None
):
    if active_tl_links is None:
        active_tl_links = set()

    graph = {}
    edge_components = {}

    for link_type, links in network['links'].items():
        for u, neighbors in links.items():
            if u not in graph:
                graph[u] = []
            for v, dist_km in neighbors:
                Tprop = dist_km / c_km_s

                # حالت کامل (مثل قبل)
                Tnode = node_delay_const

                total_delay = Tprop + Tnode

                graph[u].append((v, total_delay))
                edge_components[(u, v)] = {
                    'prop': Tprop,
                    'node': Tnode,
                    'total': total_delay,
                }

    return graph, edge_components



#  Routing Algorithm (Dijkstra Shortest Path)

This section implements the shortest-path routing algorithm
for inter-satellite communication.

In [ ]:
def dijkstra(graph, source, target):
    heap = [(0, source, [])]
    visited = set()
    costs = {source: 0}
    paths = {source: []}
    while heap:
        cost, node, path = heapq.heappop(heap)
        if node in visited:
            continue
        visited.add(node)
        path = path + [node]
        if node == target:
            return (path, cost)
        for neighbor, weight in graph.get(node, []):
            if neighbor not in visited:
                new_cost = cost + weight
                if neighbor not in costs or new_cost < costs[neighbor]:
                    costs[neighbor] = new_cost
                    paths[neighbor] = path
                    heapq.heappush(heap, (new_cost, neighbor, path))
    return ([], float('inf'))

def sum_path_delay_components(path_item, edge_components):
    path = path_item['path']
    prop_sum = node_sum = setup_sum = total_sum = 0.0
    for u, v in zip(path[:-1], path[1:]):
        comp = edge_components.get((u, v)) or edge_components.get((v, u))
        if comp:
            prop_sum += comp['prop']
            node_sum += comp['node']
            total_sum += comp['total']
    return {
        'propagation_s': prop_sum,
        'node_s': node_sum,
        'setup_s': setup_sum,
        'total_s': total_sum,
        'hops': len(path) - 1    }


#  Ground Station to Satellite Mapping

This section maps each ground station (GS) to its nearest satellite.

In [ ]:
def find_gs_to_satellite_links(
    df_GS,
    df_sat,
    sat_positions
):


    gs_to_sat = {}
    sat_ids = df_sat["sat_id"].values
    kd_tree = KDTree(sat_positions)

    lat_rad = np.radians(df_GS["lat_deg"].values)
    lon_rad = np.radians(df_GS["lon_deg"].values)
    r = EARTH_RADIUS + df_GS["alt_km"].values
    x = r * np.cos(lat_rad) * np.cos(lon_rad)
    y = r * np.cos(lat_rad) * np.sin(lon_rad)
    z = r * np.sin(lat_rad)
    gs_positions = np.column_stack((x, y, z))

    for i, gs_id in enumerate(df_GS["ID"].values):
        gs_pos = gs_positions[i]


        distance_km, index = kd_tree.query(gs_pos, k=1)
        sat_id = sat_ids[index]

        gs_to_sat[int(gs_id)] = {
            "sat_id": int(sat_id),
            "distance_km": float(distance_km)
        }

    return gs_to_sat

#  Traffic Generation Model

This section generates GS-to-GS traffic using a Poisson-based model
with hot and cold demand differentiation.

In [ ]:
def generate_gs_traffic_aggregated_poisson_stateless(
    df_gs,
    sparsity=0.05,
    demand_choices=(50e6,100e6, 150e6, 200e6, 250e6, 300e6),
    hot_demand_choices=(350e6, 500e6, 700e6, 850e6, 1000e6),
    hot_weight=2.5,
    seed=None
):
    """

    - multi-flow per GS pair (Poisson)
    - hot/cold-based choice lists

    """
    import numpy as np
    import pandas as pd

    rng = np.random.default_rng(seed)
    gs_ids = df_gs["ID"].values
    n_gs = len(gs_ids)
    is_hot = df_gs["is_hot"].values.astype(bool)


    mean_pairs = int(n_gs * (n_gs - 1) / 2 * sparsity)
    n_pairs = max(1, rng.poisson(lam=mean_pairs))
    n_pairs = min(n_pairs, n_gs * (n_gs - 1) // 2)


    weights = np.where(is_hot, hot_weight, 1)
    p_weights = weights / weights.sum()

    pairs_set = set()
    while len(pairs_set) < n_pairs:
        src_idx = rng.choice(n_gs, p=p_weights)
        dst_idx = rng.choice(n_gs, p=p_weights)
        if src_idx != dst_idx:
            pairs_set.add(tuple(sorted((src_idx, dst_idx))))

    pairs = np.array(list(pairs_set), dtype=int)

    # خروجی‌ها
    src_list = []
    dst_list = []
    demand_list = []
    hot_mask_list = []


    for src_idx, dst_idx in pairs:
        src_hot, dst_hot = is_hot[src_idx], is_hot[dst_idx]

        if src_hot and dst_hot:
            n_demands = max(1, rng.poisson(6))
            choices = hot_demand_choices
            hot_flag = True
        else:
            n_demands = max(1, rng.poisson(3))
            choices = demand_choices
            hot_flag = False

        demands = rng.choice(choices, size=n_demands)

        src_list.extend([int(gs_ids[src_idx])] * n_demands)
        dst_list.extend([int(gs_ids[dst_idx])] * n_demands)
        demand_list.extend(demands.tolist())
        hot_mask_list.extend([hot_flag] * n_demands)

    df_flows = pd.DataFrame({
        "src_gs": src_list,
        "dst_gs": dst_list,
        "demand_Mbps": demand_list,
        "is_hot_pair": hot_mask_list
    })
    print(f" Total Number of Generated Flows = {len(df_flows)}")

    return df_flows



#  Main Simulation Engine

This section runs the dynamic satellite network simulation
across multiple time slots and evaluates routing performance.

In [ ]:
def main_basic_simulation(
    df_GS,
    total_time_steps=5,
    delta_t=5,
    sats_per_orbit=66,
    sparsity=0.1,
    seed_base=0,
    lisl_range_km=1500,
    num_tl=5,
):
    import numpy as np
    import pandas as pd

    results = []

    for ts in range(total_time_steps):
        print(f"\n🛰️ Time Slot {ts+1}/{total_time_steps}")

        current_time = ts * delta_t

        #  Generate constellation snapshot
        df_sat = rotate_all_orbits_by_plane(
            time_minutes=current_time / 60.0
        )

        positions = spherical_to_cartesian(
            df_sat["lat_deg"].values,
            df_sat["lon_deg"].values,
            df_sat["alt_km"].values
        )

        #  Build network
        network = build_satellite_network_dynamic(
            df_sat,
            positions,
            sats_per_orbit=sats_per_orbit,
            max_tl_distance=lisl_range_km,
            num=num_tl
        )

        #  Build propagation-delay weighted graph
        graph, edge_components = build_weighted_graph(
            network
        )

        #  Map GS to nearest satellite
        gs_to_sat = find_gs_to_satellite_links(
            df_GS,
            df_sat,
            positions
        )

        #  Generate traffic (stateless)
        flows = generate_gs_traffic_aggregated_poisson_stateless(
            df_GS,
            sparsity=sparsity,
            seed=seed_base + ts
        )

        total_flows = len(flows)
        success = 0
        delays = []
        hops_list = []

        # Route flows (simple shortest path)
        for row in flows.itertuples():
            try:
                src_sat = gs_to_sat[int(row.src_gs)]["sat_id"]
                dst_sat = gs_to_sat[int(row.dst_gs)]["sat_id"]
            except KeyError:
                continue

            path, cost = dijkstra(graph, src_sat, dst_sat)

            if path:
                success += 1
                delays.append(cost)
                hops_list.append(len(path) - 1)

        blocking_rate = 100 * (1 - success / total_flows) if total_flows > 0 else 0
        avg_delay = np.mean(delays) if delays else 0
        avg_hops = np.mean(hops_list) if hops_list else 0

        print(
            f"   Total={total_flows}, Success={success}, "
            f"Blocking={blocking_rate:.2f}%, "
            f"AvgDelay={avg_delay:.6f}s, AvgHops={avg_hops:.2f}"
        )

        results.append({
            "time_slot": ts,
            "flows_total": total_flows,
            "flows_success": success,
            "blocking_rate_percent": blocking_rate,
            "avg_delay_s": avg_delay,
            "avg_hops": avg_hops,
        })

    return pd.DataFrame(results)